# 이번 AASIST 학습 목표

- 로컬 음원을 Colab에서 정상적으로 읽는가
- 모든 음원이 [64600] 크기로 들어가는가
- T4에서 AASIST가 OOM 없이 학습되는가
- Validation metric이 정상적으로 계산되는가

# 필요 라이브러리 install

In [25]:
import numpy as np
import scipy
import sklearn

print("NumPy       :", np.__version__)
print("SciPy       :", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)

NumPy       : 2.0.2
SciPy       : 1.16.3
Scikit-learn: 1.6.1


In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
# from sklearn.model_selection import train_test_split

# print("scikit-learn import 성공")

In [28]:
# GPU 확인

!nvidia-smi

import sys
import torch
import platform

print("Python :", sys.version)
print("OS     :", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.version.cuda)
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Thu Aug 20 05:55:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   33C    P8             16W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [29]:
# # 시스템 패키지 설치

# # Codec augmentation을 위해 FFmpeg를 설치합니다.

# !apt-get update -qq
# !apt-get install -y -qq ffmpeg libsndfile1

In [30]:
# ============================================================
# Cell 2
# 필요한 추가 패키지만 설치
# ============================================================

!apt-get update -qq
!apt-get install -y -qq libsndfile1 ffmpeg

# Colab에 기본적으로 없는/버전 확인이 필요한 것만
%pip install -q soundfile librosa

# AASIST 공식 코드
!rm -rf /content/aasist

!git clone -q \
    https://github.com/clovaai/aasist.git \
    /content/aasist

print("설치 완료")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
설치 완료


In [31]:
# %pip install -q -U \
#     numpy \
#     pandas \
#     scipy \
#     scikit-learn \
#     librosa \
#     soundfile \
#     soxr \
#     matplotlib \
#     tqdm \
#     pyyaml \
#     joblib \
#     einops \
#     tensorboard

In [32]:
# # XLS-R용 Hugging Face 설치
# %pip install -q -U \
#     transformers \
#     accelerate \
#     datasets \
#     huggingface_hub \
#     safetensors

In [33]:
# XLSR_MODEL_NAME = "facebook/wav2vec2-xls-r-300m"

In [34]:
# # AASIST 설치

# !git clone -q https://github.com/clovaai/aasist.git /content/aasist

In [35]:
# # RawBoost 설치

# !git clone -q \
#     https://github.com/TakHemlata/RawBoost-antispoofing.git \
#     /content/RawBoost-antispoofing

In [36]:
# # mamba 설치
# %pip install -q -U ninja packaging
# %pip install -q "mamba-ssm[causal-conv1d]" --no-build-isolation

In [37]:
# ============================================================
# Cell 3
# Library Import
# ============================================================

import os
import sys
import gc
import json
import math
import random
import shutil
import subprocess

from pathlib import Path
from collections import Counter


# ============================================================
# Data
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# Audio
# ============================================================

import soundfile as sf


# ============================================================
# PyTorch
# ============================================================

import torch
import torch.nn as nn

from torch.utils.data import (
    DataLoader,
)


# ============================================================
# sklearn
# ============================================================

import sklearn

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
)


# ============================================================
# Utility
# ============================================================

from tqdm.auto import tqdm

import matplotlib.pyplot as plt


# ============================================================
# Device
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


print("=" * 70)

print("NumPy        :", np.__version__)
print("Pandas       :", pd.__version__)
print("Scikit-learn :", sklearn.__version__)
print("PyTorch      :", torch.__version__)
print("Device       :", DEVICE)

if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))

print("=" * 70)

NumPy        : 2.0.2
Pandas       : 2.2.3
Scikit-learn : 1.6.1
PyTorch      : 2.11.0+cu128
Device       : cuda
GPU          : NVIDIA L4


In [38]:
# AASIST IMPORT

!rm -rf /content/aasist

!git clone -q \
    https://github.com/clovaai/aasist.git \
    /content/aasist

print("AASIST repository clone 완료")

AASIST repository clone 완료


In [39]:
# Seed설정

SEED = 42


def seed_everything(seed=42):

    random.seed(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # 재현성 중심
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)

print("✅ Seed =", SEED)

✅ Seed = 42


In [40]:
# # datasets zip 파일 업로드 및 압축해제

# from pathlib import Path

# ZIP_PATH = Path(
#     "/content/drive/MyDrive/Dacon 경진대회/asvspoof2019/LA.zip"
# )

# print("ZIP_PATH :", ZIP_PATH)
# print("존재 여부 :", ZIP_PATH.exists())

In [41]:
import shutil

total, used, free = shutil.disk_usage("/content")

GB = 1024 ** 3

print(f"전체 공간 : {total / GB:.2f} GB")
print(f"사용 공간 : {used / GB:.2f} GB")
print(f"남은 공간 : {free / GB:.2f} GB")

전체 공간 : 235.68 GB
사용 공간 : 61.74 GB
남은 공간 : 173.92 GB


In [42]:
# # DRIVE의 ZIP파일을 CONTENT로 복사
# import shutil
# from pathlib import Path

# ZIP_PATH = Path(
#     "/content/drive/MyDrive/Dacon 경진대회/asvspoof2019/LA.zip"
# )

# LOCAL_ZIP = Path(
#     "/content/dataset.zip"
# )

# print("Google Drive → Colab 복사 시작")

# shutil.copy2(
#     ZIP_PATH,
#     LOCAL_ZIP
# )

# print("복사 완료")

# print(
#     f"ZIP 크기 : "
#     f"{LOCAL_ZIP.stat().st_size / 1024**3:.2f} GB"
# )

In [43]:
# # COLAB 로컬에서 압축해제
# import zipfile
# from pathlib import Path

# LOCAL_ZIP = Path(
#     "/content/dataset.zip"
# )

# DATA_DIR = Path(
#     "/content/data"
# )

# DATA_DIR.mkdir(
#     parents=True,
#     exist_ok=True
# )

# print("압축 해제 시작")

# with zipfile.ZipFile(
#     LOCAL_ZIP,
#     "r"
# ) as zip_ref:

#     zip_ref.extractall(
#         DATA_DIR
#     )

# print("압축 해제 완료")

In [44]:
# LOCAL_ZIP.unlink()

# print("Colab의 ZIP 파일 삭제 완료")

In [45]:
# from pathlib import Path

# DATA_DIR = Path("/content/data")

# print("=== 최상위 구조 ===")

# for p in sorted(DATA_DIR.iterdir()):
#     print(
#         "[DIR] " if p.is_dir() else "[FILE]",
#         p
#     )

In [46]:
# print("\n=== 하위 폴더 구조 ===")

# for p in sorted(DATA_DIR.rglob("*")):
#     if p.is_dir():
#         print(p)

In [47]:
from pathlib import Path

DATA_ROOT = Path(
    "/content/data/LA"
)

TRAIN_DIR = (
    DATA_ROOT
    / "ASVspoof2019_LA_train"
)

DEV_DIR = (
    DATA_ROOT
    / "ASVspoof2019_LA_dev"
)

EVAL_DIR = (
    DATA_ROOT
    / "ASVspoof2019_LA_eval"
)

PROTOCOL_DIR = (
    DATA_ROOT
    / "ASVspoof2019_LA_cm_protocols"
)


TRAIN_PROTOCOL = (
    PROTOCOL_DIR
    / "ASVspoof2019.LA.cm.train.trn.txt"
)

DEV_PROTOCOL = (
    PROTOCOL_DIR
    / "ASVspoof2019.LA.cm.dev.trl.txt"
)

EVAL_PROTOCOL = (
    PROTOCOL_DIR
    / "ASVspoof2019.LA.cm.eval.trl.txt"
)


print("DATA_ROOT")
print(DATA_ROOT)

print("\nTRAIN")
print(TRAIN_DIR)

print("\nDEV")
print(DEV_DIR)

print("\nEVAL")
print(EVAL_DIR)

print("\nTRAIN Protocol")
print(TRAIN_PROTOCOL)

DATA_ROOT
/content/data/LA

TRAIN
/content/data/LA/ASVspoof2019_LA_train

DEV
/content/data/LA/ASVspoof2019_LA_dev

EVAL
/content/data/LA/ASVspoof2019_LA_eval

TRAIN Protocol
/content/data/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt


In [48]:
# 경로가 제대로 잡혔는가
paths = {
    "DATA_ROOT": DATA_ROOT,
    "TRAIN_DIR": TRAIN_DIR,
    "DEV_DIR": DEV_DIR,
    "EVAL_DIR": EVAL_DIR,
    "TRAIN_PROTOCOL": TRAIN_PROTOCOL,
    "DEV_PROTOCOL": DEV_PROTOCOL,
    "EVAL_PROTOCOL": EVAL_PROTOCOL,
}

for name, path in paths.items():

    print(
        f"{name:20s}",
        path.exists(),
        path
    )

DATA_ROOT            True /content/data/LA
TRAIN_DIR            True /content/data/LA/ASVspoof2019_LA_train
DEV_DIR              True /content/data/LA/ASVspoof2019_LA_dev
EVAL_DIR             True /content/data/LA/ASVspoof2019_LA_eval
TRAIN_PROTOCOL       True /content/data/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt
DEV_PROTOCOL         True /content/data/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt
EVAL_PROTOCOL        True /content/data/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt


In [49]:
# flac 갯수확인
train_flacs = list(
    (TRAIN_DIR / "flac").glob("*.flac")
)

dev_flacs = list(
    (DEV_DIR / "flac").glob("*.flac")
)

eval_flacs = list(
    (EVAL_DIR / "flac").glob("*.flac")
)


print(
    "Train FLAC:",
    len(train_flacs)
)

print(
    "Dev FLAC:",
    len(dev_flacs)
)

print(
    "Eval FLAC:",
    len(eval_flacs)
)

Train FLAC: 25380
Dev FLAC: 24986
Eval FLAC: 71933


In [50]:
# audio sample Rate 확인

for audio_path in train_flacs[:5]:

    info = sf.info(
        str(audio_path)
    )

    print(
        audio_path.name,
        "| SR:",
        info.samplerate,
        "| Frames:",
        info.frames,
        "| Duration:",
        round(info.duration, 3),
        "sec"
    )

LA_T_3167607.flac | SR: 16000 | Frames: 31238 | Duration: 1.952 sec
LA_T_4043541.flac | SR: 16000 | Frames: 78428 | Duration: 4.902 sec
LA_T_4607416.flac | SR: 16000 | Frames: 71579 | Duration: 4.474 sec
LA_T_2671174.flac | SR: 16000 | Frames: 45202 | Duration: 2.825 sec
LA_T_8533456.flac | SR: 16000 | Frames: 67289 | Duration: 4.206 sec


In [51]:
# protocol 내용 확인
with open(
    TRAIN_PROTOCOL,
    "r"
) as f:

    for _ in range(10):

        line = f.readline().strip()

        print(line)

LA_0079 LA_T_1138215 - - bonafide
LA_0079 LA_T_1271820 - - bonafide
LA_0079 LA_T_1272637 - - bonafide
LA_0079 LA_T_1276960 - - bonafide
LA_0079 LA_T_1341447 - - bonafide
LA_0079 LA_T_1363611 - - bonafide
LA_0079 LA_T_1596451 - - bonafide
LA_0079 LA_T_1608170 - - bonafide
LA_0079 LA_T_1684951 - - bonafide
LA_0079 LA_T_1699801 - - bonafide


In [52]:
# 각 항목을 분해해서 확인
with open(
    TRAIN_PROTOCOL,
    "r"
) as f:

    line = f.readline().strip()


parts = line.split()


print("전체:", parts)
print()

print(
    "Speaker ID :",
    parts[0]
)

print(
    "Utterance ID:",
    parts[1]
)

print(
    "Attack ID :",
    parts[3]
)

print(
    "Label     :",
    parts[4]
)

전체: ['LA_0079', 'LA_T_1138215', '-', '-', 'bonafide']

Speaker ID : LA_0079
Utterance ID: LA_T_1138215
Attack ID : -
Label     : bonafide


In [53]:
# dataframe 만들기
import pandas as pd


def protocol_to_dataframe(
    protocol_path,
    audio_dir,
    split_name
):

    records = []


    with open(
        protocol_path,
        "r"
    ) as f:

        for line in f:

            parts = (
                line
                .strip()
                .split()
            )


            if len(parts) != 5:
                continue


            speaker_id = parts[0]

            utterance_id = parts[1]

            attack_id = parts[3]

            label_text = parts[4]


            # AASIST 공식 convention
            #
            # bonafide = 1
            # spoof    = 0

            label = (
                1
                if label_text == "bonafide"
                else 0
            )


            audio_path = (
                audio_dir
                / "flac"
                / f"{utterance_id}.flac"
            )


            records.append(
                {
                    "speaker_id":
                        speaker_id,

                    "utterance_id":
                        utterance_id,

                    "attack_id":
                        attack_id,

                    "label_text":
                        label_text,

                    "label":
                        label,

                    "path":
                        str(audio_path),

                    "split":
                        split_name,
                }
            )


    return pd.DataFrame(
        records
    )

In [54]:
# train dataframe 생성
train_df = protocol_to_dataframe(
    TRAIN_PROTOCOL,
    TRAIN_DIR,
    "train",
)


print(
    "Train shape:",
    train_df.shape
)


display(
    train_df.head(10)
)

Train shape: (25380, 7)


,speaker_id,utterance_id,attack_id,label_text,label,path,split
0,LA_0079,LA_T_1138215,-,bonafide,1,/content/data/LA/ASVspoof2019_LA_train/flac/LA...,train
1,LA_0079,LA_T_1271820,-,bonafide,1,/content/data/LA/ASVspoof2019_LA_train/flac/LA...,train
2,LA_0079,LA_T_1272637,-,bonafide,1,/content/data/LA/ASVspoof2019_LA_train/flac/LA...,train
3,LA_0079,LA_T_1276960,-,bonafide,1,/content/data/LA/ASVspoof2019_LA_train/flac/LA...,train
4,LA_0079,LA_T_1341447,-,bonafide,1,/content/data/LA/ASVspoof2019_LA_train/flac/LA...,train
5,LA_0079,LA_T_1363611,-,bonafide,1,/content/data/LA/ASVspoof2019_LA_train/flac/LA...,train
6,LA_0079,LA_T_1596451,-,bonafide,1,/content/data/LA/ASVspoof2019_LA_train/flac/LA...,train
7,LA_0079,LA_T_1608170,-,bonafide,1,/content/data/LA/ASVspoof2019_LA_train/flac/LA...,train
8,LA_0079,LA_T_1684951,-,bonafide,1,/content/data/LA/ASVspoof2019_LA_train/flac/LA...,train
9,LA_0079,LA_T_1699801,-,bonafide,1,/content/data/LA/ASVspoof2019_LA_train/flac/LA...,train


In [55]:
# label 갯수확인
print(
    train_df[
        "label_text"
    ].value_counts()
)

label_text
spoof       22800
bonafide     2580
Name: count, dtype: int64


In [56]:
# 비율확인
print(
    train_df[
        "label_text"
    ].value_counts(
        normalize=True
    )
)

label_text
spoof       0.898345
bonafide    0.101655
Name: proportion, dtype: float64


In [57]:
# 파일이 실재하는지 확인
train_df[
    "file_exists"
] = train_df[
    "path"
].apply(
    lambda x: Path(x).exists()
)


print(
    train_df[
        "file_exists"
    ].value_counts()
)

file_exists
True    25380
Name: count, dtype: int64


In [58]:
dev_df = protocol_to_dataframe(
    DEV_PROTOCOL,
    DEV_DIR,
    "dev",
)


print(
    "Dev shape:",
    dev_df.shape
)


print(
    dev_df[
        "label_text"
    ].value_counts()
)

Dev shape: (24844, 7)
label_text
spoof       22296
bonafide     2548
Name: count, dtype: int64


In [59]:
# aasist 공식 데이터함수 import

AASIST_ROOT = Path(
    "/content/aasist"
)


if str(AASIST_ROOT) not in sys.path:

    sys.path.insert(
        0,
        str(AASIST_ROOT)
    )


from data_utils import (
    genSpoof_list,
    Dataset_ASVspoof2019_train,
    Dataset_ASVspoof2019_devNeval,
)

from utils import (
    create_optimizer,
    seed_worker,
)


print("✅ AASIST data_utils import 완료")

✅ AASIST data_utils import 완료


In [63]:
# ============================================================
# Protocol 다시 읽기
# ============================================================

from data_utils import (
    genSpoof_list,
    Dataset_ASVspoof2019_train,
    Dataset_ASVspoof2019_devNeval,
)

# Train
train_labels, train_ids = genSpoof_list(
    dir_meta=TRAIN_PROTOCOL,
    is_train=True,
    is_eval=False,
)

# Dev
dev_labels, dev_ids = genSpoof_list(
    dir_meta=DEV_PROTOCOL,
    is_train=False,
    is_eval=False,
)

print("Train samples :", len(train_ids))
print("Dev samples   :", len(dev_ids))

print("첫 Train ID   :", train_ids[0])
print("첫 Train Label:", train_labels[train_ids[0]])

print("첫 Dev ID     :", dev_ids[0])
print("첫 Dev Label  :", dev_labels[dev_ids[0]])

Train samples : 25380
Dev samples   : 24844
첫 Train ID   : LA_T_1138215
첫 Train Label: 1
첫 Dev ID     : LA_D_1047731
첫 Dev Label  : 1


In [64]:
# Dataset

train_dataset = Dataset_ASVspoof2019_train(
    list_IDs=train_ids,
    labels=train_labels,
    base_dir=TRAIN_DIR,
)


dev_dataset = Dataset_ASVspoof2019_devNeval(
    list_IDs=dev_ids,
    base_dir=DEV_DIR,
)


print(
    "Train Dataset:",
    len(train_dataset)
)

print(
    "Dev Dataset  :",
    len(dev_dataset)
)

Train Dataset: 25380
Dev Dataset  : 24844


In [85]:
# ============================================================
# Training / DataLoader 설정
# ============================================================

SEED = 42

BATCH_SIZE = 8
NUM_WORKERS = 0

EPOCHS = 5

FREQ_AUG = False

print("BATCH_SIZE :", BATCH_SIZE)
print("NUM_WORKERS:", NUM_WORKERS)
print("EPOCHS     :", EPOCHS)

BATCH_SIZE : 8
NUM_WORKERS: 0
EPOCHS     : 5


In [86]:
# dataloader 생성

generator = torch.Generator()

generator.manual_seed(
    SEED
)


train_loader = DataLoader(
    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    drop_last=True,

    pin_memory=True,

    num_workers=NUM_WORKERS,

    worker_init_fn=seed_worker,

    generator=generator,

    persistent_workers=(
        NUM_WORKERS > 0
    ),
)


dev_loader = DataLoader(
    dev_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    drop_last=False,

    pin_memory=True,

    num_workers=NUM_WORKERS,

    persistent_workers=(
        NUM_WORKERS > 0
    ),
)


print(
    "Train batches:",
    len(train_loader)
)

print(
    "Dev batches  :",
    len(dev_loader)
)

Train batches: 3172
Dev batches  : 3106


In [87]:
batch_x, batch_y = next(
    iter(train_loader)
)

print("Audio Shape :", batch_x.shape)
print("Label Shape :", batch_y.shape)
print("Labels      :", batch_y)

Audio Shape : torch.Size([8, 64600])
Label Shape : torch.Size([8])
Labels      : tensor([1, 0, 0, 0, 0, 0, 0, 0])


In [72]:
# dataloader 확인
# ============================================================
# Cell 20
# Batch Sanity Check
# ============================================================

batch_x, batch_y = next(
    iter(train_loader)
)


print(
    "Audio shape:",
    batch_x.shape
)

print(
    "Label shape:",
    batch_y.shape
)

print(
    "Labels:",
    batch_y
)

print(
    "Audio dtype:",
    batch_x.dtype
)

Audio shape: torch.Size([8, 64600])
Label shape: torch.Size([8])
Labels: tensor([1, 0, 0, 0, 0, 0, 0, 0])
Audio dtype: torch.float32


In [88]:

# Official AASIST Config 확인


CONFIG_PATH = (
    AASIST_ROOT
    / "config"
    / "AASIST.conf"
)


with open(
    CONFIG_PATH,
    "r"
) as f:

    config = json.load(f)


model_config = config[
    "model_config"
]


optim_config = config[
    "optim_config"
].copy()


print(
    json.dumps(
        model_config,
        indent=2
    )
)

{
  "architecture": "AASIST",
  "nb_samp": 64600,
  "first_conv": 128,
  "filts": [
    70,
    [
      1,
      32
    ],
    [
      32,
      32
    ],
    [
      32,
      64
    ],
    [
      64,
      64
    ]
  ],
  "gat_dims": [
    64,
    32
  ],
  "pool_ratios": [
    0.5,
    0.7,
    0.5,
    0.5
  ],
  "temperatures": [
    2.0,
    2.0,
    100.0,
    100.0
  ]
}


In [89]:
# AASIST Model 생성

from models.AASIST import (
    Model as AASIST
)


model = AASIST(
    model_config
)


model = model.to(
    DEVICE
)


total_params = sum(
    p.numel()
    for p in model.parameters()
)


trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(
    "Total Parameters    :",
    f"{total_params:,}"
)

print(
    "Trainable Parameters:",
    f"{trainable_params:,}"
)

Exception ignored in: <function tqdm.__del__ at 0x7cf7dc1a6a20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/usr/local/lib/python3.12/dist-packages/tqdm/notebook.py", line 277, in close
    self.disp(bar_style='danger', check_delay=False)
    ^^^^^^^^^
AttributeError: 'tqdm' object has no attribute 'disp'


Total Parameters    : 297,866
Trainable Parameters: 297,866


In [90]:
# Forward Test

model.eval()


batch_x, batch_y = next(
    iter(train_loader)
)


batch_x = batch_x[:2].to(
    DEVICE
)


with torch.no_grad():

    hidden, logits = model(
        batch_x,
        Freq_aug=False
    )


print(
    "Input :",
    batch_x.shape
)

print(
    "Hidden:",
    hidden.shape
)

print(
    "Logits:",
    logits.shape
)

Input : torch.Size([2, 64600])
Hidden: torch.Size([2, 160])
Logits: torch.Size([2, 2])


In [91]:
# Loss / Optimizer / Scheduler

class_weights = torch.tensor(
    [0.1, 0.9],
    dtype=torch.float32,
    device=DEVICE,
)


criterion = nn.CrossEntropyLoss(
    weight=class_weights
)


# 우리가 사용할 epoch 수 반영
optim_config["epochs"] = EPOCHS

optim_config[
    "steps_per_epoch"
] = len(train_loader)


optimizer, scheduler = create_optimizer(
    model.parameters(),
    optim_config,
)


print(
    "Optimizer:",
    optimizer
)

print(
    "Scheduler:",
    scheduler
)

print(
    "Learning Rate:",
    optim_config["base_lr"]
)

Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.0001
    lr: 0.0001
    maximize: False
    weight_decay: 0.0001
)
Scheduler: <torch.optim.lr_scheduler.LambdaLR object at 0x7cf7ccd59040>
Learning Rate: 0.0001


In [92]:
# EER
def calculate_eer(
    y_true,
    scores
):

    fpr, tpr, thresholds = roc_curve(
        y_true,
        scores,
        pos_label=1,
    )


    fnr = 1.0 - tpr


    idx = np.nanargmin(
        np.abs(
            fpr - fnr
        )
    )


    eer = (
        fpr[idx]
        +
        fnr[idx]
    ) / 2


    threshold = thresholds[
        idx
    ]


    return (
        float(eer),
        float(threshold)
    )

In [93]:
# Train One Epoch

def train_one_epoch(
    model,
    loader,
    optimizer,
    scheduler,
    criterion,
    device,
):

    model.train()


    running_loss = 0.0

    total_samples = 0


    predictions = []

    targets = []


    progress = tqdm(
        loader,
        desc="TRAIN",
        leave=False,
    )


    for audio, label in progress:

        audio = audio.to(
            device,
            non_blocking=True
        )


        label = (
            label
            .view(-1)
            .long()
            .to(
                device,
                non_blocking=True
            )
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        _, logits = model(
            audio,
            Freq_aug=FREQ_AUG
        )


        loss = criterion(
            logits,
            label
        )


        loss.backward()


        optimizer.step()


        # 공식 AASIST cosine scheduler는
        # batch 단위로 step
        if scheduler is not None:

            scheduler.step()


        batch_size = audio.size(0)


        running_loss += (
            loss.item()
            *
            batch_size
        )


        total_samples += (
            batch_size
        )


        pred = logits.argmax(
            dim=1
        )


        predictions.extend(
            pred.detach()
            .cpu()
            .numpy()
        )


        targets.extend(
            label.detach()
            .cpu()
            .numpy()
        )


        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )


    epoch_loss = (
        running_loss
        /
        total_samples
    )


    accuracy = accuracy_score(
        targets,
        predictions
    )


    return (
        epoch_loss,
        accuracy
    )

In [94]:
# ============================================================
# Cell 27
# Validation
# ============================================================

def validate(
    model,
    loader,
    label_dict,
    criterion,
    device,
):

    model.eval()


    running_loss = 0.0

    total_samples = 0


    all_labels = []

    all_preds = []

    all_scores = []

    all_ids = []


    with torch.no_grad():

        progress = tqdm(
            loader,
            desc="DEV",
            leave=False,
        )


        for audio, utt_ids in progress:

            audio = audio.to(
                device,
                non_blocking=True
            )


            labels = torch.tensor(
                [
                    label_dict[
                        utt_id
                    ]
                    for utt_id
                    in utt_ids
                ],
                dtype=torch.long,
                device=device,
            )


            _, logits = model(
                audio,
                Freq_aug=False
            )


            loss = criterion(
                logits,
                labels
            )


            # 공식 AASIST와 동일:
            # 두 번째 출력 = bona fide score
            bona_scores = (
                logits[:, 1]
                .detach()
                .cpu()
                .numpy()
            )


            predictions = (
                logits.argmax(
                    dim=1
                )
                .detach()
                .cpu()
                .numpy()
            )


            batch_size = audio.size(
                0
            )


            running_loss += (
                loss.item()
                *
                batch_size
            )


            total_samples += (
                batch_size
            )


            all_labels.extend(
                labels.cpu().numpy()
            )


            all_preds.extend(
                predictions
            )


            all_scores.extend(
                bona_scores
            )


            all_ids.extend(
                list(utt_ids)
            )


    all_labels = np.asarray(
        all_labels
    )

    all_preds = np.asarray(
        all_preds
    )

    all_scores = np.asarray(
        all_scores
    )


    val_loss = (
        running_loss
        /
        total_samples
    )


    accuracy = accuracy_score(
        all_labels,
        all_preds
    )


    balanced_acc = (
        balanced_accuracy_score(
            all_labels,
            all_preds
        )
    )


    f1 = f1_score(
        all_labels,
        all_preds,
        pos_label=1,
    )


    auc = roc_auc_score(
        all_labels,
        all_scores
    )


    eer, eer_threshold = (
        calculate_eer(
            all_labels,
            all_scores
        )
    )


    return {

        "loss":
            val_loss,

        "accuracy":
            accuracy,

        "balanced_accuracy":
            balanced_acc,

        "f1":
            f1,

        "auc":
            auc,

        "eer":
            eer,

        "eer_threshold":
            eer_threshold,

        "labels":
            all_labels,

        "preds":
            all_preds,

        "scores":
            all_scores,

        "ids":
            all_ids,
    }

In [95]:
# ============================================================
# Checkpoint 저장 경로 설정
# ============================================================

from pathlib import Path
import os

# Google Drive가 이미 mount 되어 있다고 가정
DRIVE_PROJECT_DIR = Path(
    "/content/drive/MyDrive/Dacon/DeepVoice"
)

# 모델 checkpoint 저장 폴더
DRIVE_CHECKPOINT_DIR = (
    DRIVE_PROJECT_DIR
    / "checkpoints"
)

# 폴더가 없으면 자동 생성
DRIVE_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Best Model 저장 경로
BEST_MODEL_PATH = (
    DRIVE_CHECKPOINT_DIR
    / "aasist_baseline_best.pt"
)


# 학습 기록 CSV 저장 경로
HISTORY_PATH = (
    DRIVE_CHECKPOINT_DIR
    / "aasist_baseline_history.csv"
)


print("=" * 70)
print("Project Dir    :", DRIVE_PROJECT_DIR)
print("Checkpoint Dir :", DRIVE_CHECKPOINT_DIR)
print("Best Model     :", BEST_MODEL_PATH)
print("History        :", HISTORY_PATH)
print("=" * 70)

print(
    "Checkpoint Dir 존재:",
    DRIVE_CHECKPOINT_DIR.exists()
)

Project Dir    : /content/drive/MyDrive/Dacon/DeepVoice
Checkpoint Dir : /content/drive/MyDrive/Dacon/DeepVoice/checkpoints
Best Model     : /content/drive/MyDrive/Dacon/DeepVoice/checkpoints/aasist_baseline_best.pt
History        : /content/drive/MyDrive/Dacon/DeepVoice/checkpoints/aasist_baseline_history.csv
Checkpoint Dir 존재: True


In [ ]:
# 모델학습

BEST_MODEL_PATH = (
    DRIVE_CHECKPOINT_DIR
    / "aasist_baseline_best.pt"
)


HISTORY_PATH = (
    DRIVE_CHECKPOINT_DIR
    / "aasist_baseline_history.csv"
)


best_eer = float(
    "inf"
)


history = []


for epoch in range(
    1,
    EPOCHS + 1
):

    print()
    print("=" * 70)
    print(
        f"EPOCH "
        f"{epoch}/{EPOCHS}"
    )
    print("=" * 70)


    # ========================================================
    # Train
    # ========================================================

    train_loss, train_acc = (
        train_one_epoch(
            model,
            train_loader,
            optimizer,
            scheduler,
            criterion,
            DEVICE,
        )
    )


    # ========================================================
    # Dev
    # ========================================================

    dev_result = validate(
        model,
        dev_loader,
        dev_labels,
        criterion,
        DEVICE,
    )


    print(
        f"Train Loss      : "
        f"{train_loss:.5f}"
    )

    print(
        f"Train Accuracy  : "
        f"{train_acc:.5f}"
    )

    print()

    print(
        f"Dev Loss        : "
        f"{dev_result['loss']:.5f}"
    )

    print(
        f"Dev Accuracy    : "
        f"{dev_result['accuracy']:.5f}"
    )

    print(
        f"Dev Balanced Acc: "
        f"{dev_result['balanced_accuracy']:.5f}"
    )

    print(
        f"Dev F1          : "
        f"{dev_result['f1']:.5f}"
    )

    print(
        f"Dev AUC         : "
        f"{dev_result['auc']:.5f}"
    )

    print(
        f"Dev EER         : "
        f"{dev_result['eer'] * 100:.3f}%"
    )


    # ========================================================
    # History
    # ========================================================

    history.append(
        {
            "epoch":
                epoch,

            "train_loss":
                train_loss,

            "train_accuracy":
                train_acc,

            "dev_loss":
                dev_result["loss"],

            "dev_accuracy":
                dev_result["accuracy"],

            "dev_balanced_accuracy":
                dev_result[
                    "balanced_accuracy"
                ],

            "dev_f1":
                dev_result["f1"],

            "dev_auc":
                dev_result["auc"],

            "dev_eer":
                dev_result["eer"],
        }
    )


    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_PATH,
        index=False,
    )


    # ========================================================
    # Best Model
    # ========================================================

    if (
        dev_result["eer"]
        <
        best_eer
    ):

        best_eer = (
            dev_result[
                "eer"
            ]
        )


        torch.save(
            {
                "epoch":
                    epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "scheduler_state_dict":
                    (
                        scheduler.state_dict()
                        if scheduler
                        is not None
                        else None
                    ),

                "dev_eer":
                    best_eer,

                "model_config":
                    model_config,

                "optim_config":
                    optim_config,
            },
            BEST_MODEL_PATH,
        )


        print()
        print(
            "⭐ Best Model 저장"
        )

        print(
            "EER:",
            f"{best_eer * 100:.3f}%"
        )

        print(
            "Path:",
            BEST_MODEL_PATH
        )


    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


EPOCH 1/5


TRAIN:   0%|          | 0/3172 [00:00<?, ?it/s]

DEV:   0%|          | 0/3106 [00:00<?, ?it/s]

Train Loss      : 0.50180
Train Accuracy  : 0.81215

Dev Loss        : 0.38436
Dev Accuracy    : 0.79569
Dev Balanced Acc: 0.88269
Dev F1          : 0.49901
Dev AUC         : 0.97925
Dev EER         : 6.741%

⭐ Best Model 저장
EER: 6.741%
Path: /content/drive/MyDrive/Dacon/DeepVoice/checkpoints/aasist_baseline_best.pt

EPOCH 2/5


TRAIN:   0%|          | 0/3172 [00:00<?, ?it/s]

...

Train Loss      : 0.21985
Train Accuracy  : 0.92599

Dev Loss        : 0.18288
Dev Accuracy    : 0.92075
Dev Balanced Acc: 0.94368
Dev F1          : 0.71567
Dev AUC         : 0.98841
Dev EER         : 5.171%

⭐ Best Model 저장
EER: 5.171%
Path: /content/drive/MyDrive/Dacon/DeepVoice/checkpoints/aasist_baseline_best.pt

EPOCH 3/5


TRAIN:   0%|          | 0/3172 [00:00<?, ?it/s]

DEV:   0%|          | 0/3106 [00:00<?, ?it/s]

Train Loss      : 0.14891
Train Accuracy  : 0.95468

Dev Loss        : 0.10412
Dev Accuracy    : 0.95910
Dev Balanced Acc: 0.97113
Dev F1          : 0.83184
Dev AUC         : 0.99561
Dev EER         : 2.755%

⭐ Best Model 저장
EER: 2.755%
Path: /content/drive/MyDrive/Dacon/DeepVoice/checkpoints/aasist_baseline_best.pt

EPOCH 4/5


TRAIN:   0%|          | 0/3172 [00:00<?, ?it/s]

In [ ]:
# Training Histor

history_df = pd.DataFrame(
    history
)


display(
    history_df
)

In [ ]:
# ============================================================
# Cell 30
# Loss Graph
# ============================================================

plt.figure(
    figsize=(8, 5)
)


plt.plot(
    history_df["epoch"],
    history_df["train_loss"],
    marker="o",
    label="Train"
)


plt.plot(
    history_df["epoch"],
    history_df["dev_loss"],
    marker="o",
    label="Dev"
)


plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Loss"
)

plt.title(
    "AASIST Training Loss"
)

plt.legend()

plt.grid()

plt.show()

In [ ]:
# ============================================================
# Cell 31
# EER Graph
# ============================================================

plt.figure(
    figsize=(8, 5)
)


plt.plot(
    history_df["epoch"],
    history_df["dev_eer"] * 100,
    marker="o",
)


plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "EER (%)"
)

plt.title(
    "AASIST Dev EER"
)

plt.grid()

plt.show()

In [ ]:
# ============================================================
# Cell 32
# Best Checkpoint Load
# ============================================================

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
)


model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)


print(
    "Best Epoch:",
    checkpoint["epoch"]
)

print(
    "Best Dev EER:",
    f"{checkpoint['dev_eer'] * 100:.3f}%"
)

In [ ]:
# ============================================================
# Cell 33
# Final Dev Evaluation
# ============================================================

final_result = validate(
    model,
    dev_loader,
    dev_labels,
    criterion,
    DEVICE,
)


print("=" * 70)

print(
    "AASIST BASELINE"
)

print("=" * 70)


print(
    f"Loss              : "
    f"{final_result['loss']:.5f}"
)

print(
    f"Accuracy          : "
    f"{final_result['accuracy']:.5f}"
)

print(
    f"Balanced Accuracy : "
    f"{final_result['balanced_accuracy']:.5f}"
)

print(
    f"F1                : "
    f"{final_result['f1']:.5f}"
)

print(
    f"ROC-AUC           : "
    f"{final_result['auc']:.5f}"
)

print(
    f"EER               : "
    f"{final_result['eer'] * 100:.3f}%"
)

print(
    f"EER Threshold     : "
    f"{final_result['eer_threshold']:.5f}"
)

print("=" * 70)

In [ ]:
# ============================================================
# Cell 34
# Confusion Matrix
# ============================================================

cm = confusion_matrix(
    final_result[
        "labels"
    ],
    final_result[
        "preds"
    ],
)


print(
    "Label convention"
)

print(
    "0 = SPOOF / FAKE"
)

print(
    "1 = BONAFIDE / REAL"
)

print()

print(
    cm
)

In [ ]:
print(
    classification_report(
        final_result[
            "labels"
        ],
        final_result[
            "preds"
        ],

        target_names=[
            "SPOOF",
            "BONAFIDE"
        ],

        digits=4,
    )
)